# Chapter 5 - Lab 1: Building the Deep Search Planner (customer-care edition)

**Goal**: build the **planner**, the first stage of a Deep Search Agent. Given a research question,
the planner produces a *structured research plan*: sub-tasks with explicit dependencies, ready to execute.

*Finance edition*: "Compare NVIDIA, AMD and Intel on growth and margins".
*Customer-care edition*: "Why did FrostLine refrigerator CSAT fall in Q3? Compare with the ZX1 phone and
the AquaPure purifier." The same skill a CX analyst uses: break a fuzzy business question into
answerable, parallel data requests.

You will see how **PydanticAI structured output** removes a class of problems: you declare a Pydantic
schema and the framework guarantees the model returns exactly that shape.

**Tech stack**:
* **PydanticAI** - structured-output agents.
* **OpenAI** `gpt-4o` - decomposition needs strong reasoning.
* **Pydantic v2** - typed `SubTask` and `ResearchPlan`.

## 1. Install packages

In [ ]:
%pip install -q 'pydantic-ai-slim[openai]' pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared models and helpers

Chapter 5 shares `common.py` (Pydantic models and helpers), next to this notebook in the repository.

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%205/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import ResearchPlan, SubTask, TaskStatus, PLANNING_MODEL
print("Shared models loaded. Planning model:", PLANNING_MODEL)

## 4. The planner system prompt

The planner is mostly its system prompt: which **data sources** exist (support analytics, internal CX
documents, web/social search), what makes a good plan, and the rules. Read it carefully: it is the heart
of the planner and the main lever a business team can tune.

In [ ]:
PLANNER_SYSTEM_PROMPT = """\
You are a customer-experience research planner at a retailer. Given a research
question, decompose it into concrete sub-tasks that can be executed by tools.

Available data sources for each sub-task:
- support_analytics: Structured service metrics per product line (orders, tickets,
  contact rate, CSAT and its change, first-contact resolution, repeat-contact rate,
  resolution time, return rate). Best for quantitative data.
- cx_documents: Internal documents per product line (engineering known-issue bulletins,
  field-service reports, QA call audits). Best for root causes.
- web_search: Social media, review sites, consumer forums, competitor moves.
  Best for external sentiment and market context.

Rules for creating the plan:
1. Each sub-task must have a clear, measurable outcome (e.g., "Fetch FL-340 CSAT,
   contact rate and FCR for Q3" not "Understand the fridge problem").
2. Name the product-line code (e.g. FL-340, ZX1, AQP-500) in every product-specific task.
3. Specify which data_sources each sub-task should use (can be multiple).
4. Mark dependencies: which task IDs must complete before this task can start.
5. Tasks with no dependencies can execute in parallel: maximise parallelism.
6. For multi-product comparisons, create parallel tasks for each product line.
7. Include a comparative analysis task that depends on all product data tasks.
8. The final task should synthesise all findings: it depends on everything.
9. Use sequential IDs starting from 1.
10. Aim for 5-10 sub-tasks. Too few = insufficient depth. Too many = overhead.
11. If the question is about a single product line, still gather a comparison line.\
"""

## 5. The planner agent (with structured output)

`output_type=ResearchPlan`: PydanticAI generates the JSON schema, validates the reply, and re-prompts up to `retries` times if it is invalid.

In [ ]:
from pydantic_ai import Agent

planner_agent = Agent(
    model=PLANNING_MODEL,
    system_prompt=PLANNER_SYSTEM_PROMPT,
    output_type=ResearchPlan,
    retries=2,
)

## 6. Generating a plan

In [ ]:
async def create_research_plan(question: str, additional_context: str = "") -> ResearchPlan:
    user_message = question
    if additional_context:
        user_message += f"\n\nAdditional context:\n{additional_context}"
    result = await planner_agent.run(user_message)
    return result.output

In [ ]:
question = (
    "Why did customer satisfaction for the FrostLine refrigerator line (FL-340) fall in Q3 2026? "
    "Compare it with the Zenith X1 smartphone (ZX1) and the AquaPure purifier (AQP-500) on CSAT, "
    "contact rate, first-contact resolution and repeat-contact rate, identify root causes and risks."
)

plan = await create_research_plan(question)
for task in plan.sub_tasks:
    deps = f" (depends on: {task.dependencies})" if task.dependencies else ""
    print(f"[{task.id}] {task.description}\n     sources: {', '.join(task.data_sources)}{deps}")

## 7. Validating the plan

PydanticAI guarantees the *shape* of the plan. A structurally valid plan can still be *semantically*
broken: missing dependencies, a task depending on itself, or a deadlock where every task waits for
another. A small deterministic validator catches these.

In [ ]:
def validate_plan(plan: ResearchPlan) -> None:
    """Check that the plan is semantically sound (PydanticAI already checked its shape).

    - At least 2 sub-tasks
    - All dependency IDs reference existing tasks
    - No task depends on itself
    - At least one task has no dependencies (something can start)
    """
    task_ids = {t.id for t in plan.sub_tasks}
    if len(plan.sub_tasks) < 2:
        raise ValueError(f"Plan has only {len(plan.sub_tasks)} sub-tasks. A research plan needs at least 2.")
    for task in plan.sub_tasks:
        for dep in task.dependencies:
            if dep not in task_ids:
                raise ValueError(f"Task {task.id} depends on non-existent task {dep}")
            if dep == task.id:
                raise ValueError(f"Task {task.id} depends on itself (circular dependency)")
    if not [t for t in plan.sub_tasks if not t.dependencies]:
        raise ValueError("No tasks can start - all have dependencies (deadlock)")

In [ ]:
validate_plan(plan)
print("Plan validates cleanly.")

## 8. Inspecting parallelism

In [ ]:
parallel = [t for t in plan.sub_tasks if not t.dependencies]
sequential = [t for t in plan.sub_tasks if t.dependencies]
print(f"Parallel-ready tasks: {len(parallel)}")
for t in parallel:
    print(f"  [{t.id}] {t.description[:80]}")
print(f"\nSequential tasks: {len(sequential)}")
for t in sequential:
    print(f"  [{t.id}] depends on {t.dependencies}: {t.description[:60]}")

## 9. Results

Expect 5-10 sub-tasks: one metrics task per product line (FL-340, ZX1, AQP-500), internal-document and
social-media tasks for FrostLine, a comparison task that depends on them, and a final synthesis.

**What to notice**
* **Structured output is transformative**: no parsing code; the model returns a typed object.
* **Two layers of validation**: the schema (PydanticAI) and the semantics (your `validate_plan`).
* **The prompt is the planner**: changing the list of data sources changes what research is possible.

## Discussion
1. How does this mirror how a CX analytics team scopes a "why did CSAT drop?" investigation?
2. What happens if you add a data source the company does not really have? How would you prevent that?